In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4.1-nano")
output_parser = StrOutputParser()


In [3]:
from dotenv import load_dotenv
import os
load_dotenv()
# os.getenv('OPENAI_API_KEY')

True

In [4]:
llm.invoke("Where is the capital of Korea? Answer me in Korean")

AIMessage(content='한국의 수도는 서울입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 18, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_38343a2f8f', 'id': 'chatcmpl-BmYItTUNDQu48xVne2GSmzAgefSzJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--53b63e89-f16f-4cb9-b8a5-9f1f2b430dfe-0', usage_metadata={'input_tokens': 18, 'output_tokens': 7, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# 1. 나라 -> 음식 추천

In [5]:
food_prompt = PromptTemplate(
    template="""
    Please recommend the most famous food from {country}.    
    When answering, write full name of country and,
    please use the national flag and food of the answer as an emoji.
    And give a short explanation of the name of the food and one line.

    If the user wants a specific format, follow this instruction: {format}
    """,
    input_variables=["country", "format"]
)


food_chain = food_prompt | llm | output_parser

output_parser.invoke(llm.invoke(food_prompt.invoke({"country":"Japan", "format": "Just the food name only"})))

'🇯🇵 Japan 🇯🇵  \n🍣 Sushi  \n\nSushi is a Japanese dish consisting of vinegared rice accompanied by various ingredients such as raw fish, vegetables, and sometimes tropical fruits. It is internationally recognized as a symbol of Japanese cuisine.'

# 2. 음식 -> 레시피

In [6]:
recipe_prompt = PromptTemplate(
    template="""
    Write a detailed recipe for the food: {food}.
    Include ingredients and cooking instructions.
    When answering, please use the national flag and food of the answer as an emoji.

    If the user wants a specific output style, follow this instruction: {style}
    """,
    input_variables=["food", "style"]
)


recipe_chain = recipe_prompt | llm | output_parser
output_parser.invoke(llm.invoke(recipe_prompt.invoke({"food":"kimchi", "style": "use emoji of food"})))

'🇰🇷🍽️\n\n**Kimchi Recipe**\n\n**Ingredients:**\n- 1 large Napa cabbage (about 2-3 pounds)\n- 1/4 cup sea salt or coarse salt\n- 4 cups water\n- 1 tablespoon grated ginger\n- 4 cloves garlic, minced\n- 1-2 tablespoons sugar (optional)\n- 3-4 tablespoons fish sauce or soy sauce (for vegan version)\n- 1 small carrot, julienned\n- 4 green onions, chopped\n- 1 small daikon radish, julienned\n- 2-3 tablespoons Korean red pepper flakes (gochugaru)\n- 1 teaspoon rice flour (optional, for thicker paste)\n\n**Instructions:**\n\n1. **Prepare the Cabbage:**\n   - Cut the Napa cabbage lengthwise into quarters, then roughly into 2-inch pieces.\n   - Dissolve 1/4 cup salt in 4 cups water. Submerge the cabbage pieces in the salted water, ensuring they are fully covered.\n   - Leave it to soak for 2 hours, turning occasionally for even salting.\n   \n2. **Rinse and Drain:**\n   - After 2 hours, rinse the cabbage thoroughly under cold water to remove excess salt.\n   - Drain well and set aside.\n\n3. **

# 3. 체인 연결

In [7]:
from langchain_core.runnables import RunnablePassthrough
final_chain = {"country": RunnablePassthrough(), "format": RunnablePassthrough()} | {"food": food_chain
} | {"food": RunnablePassthrough(), "style": lambda x: x["format"] } | recipe_chain


# 4. 출력

In [8]:
result = final_chain.invoke({
    "country": "Japan",
    "format": "한국어로 답변해봐"
})
print(result)


🇯🇵  
**일본식 스시 만들기**  

---

### 재료 (약 4인분)  
- 밥: 일본 쌀 2컵(약 360g)  
- 스시 식초: 1/3컵(80ml)  
- 설탕: 3큰술  
- 소금: 1작은술  
- 연어, 참치, 오이, 아보카도, 김 등 다양한 신선한 재료  
- 김밥용 김(대형 김) 4장  
- 간장, 와사비, 생강절임 (선택사항)  

---

### 준비하기  

1. **쌀 씻기 및 밥짓기**  
   - 일본 쌀을 찬물에 여러 번 씻어 전분을 제거하고 물기를 빼줍니다.  
   - 쌀과 물을 1:1.2 비율로 넣고 밥을 짓거나 전기밥솥을 사용하세요.  

2. **스시 식초 만들기**  
   - 작은 냄비에 설탕, 소금, 식초를 넣고 약한 불에서 저어가며 녹입니다.  
   - 설탕과 소금이 완전히 녹으면 식혀줍니다.  

3. **밥 양념하기**  
   - 밥이 다 다 되면, 따뜻한 밥에 식초 혼합물을 골고루 섞어줍니다.  
   - 밥을 넓은 그릇에 펼쳐 식히며 조심스럽게 섞어줍니다.  

---

### 스시 만들기  

1. **김 준비**  
   - 김을 깨끗한 도마 위에 놓고 얇게 펼쳐줍니다.  

2. **밥 펴기**  
   - 손을 조금 적셔가며, 김 위에 밥을 얇게 펴줍니다. (약 1cm 두께로)  
   - 가운데 부분은 조금 덜 덮는 것도 좋습니다.  

3. **재료 올리기**  
   - 좋아하는 재료(연어, 참치, 오이, 아보카도 등)를 밥 위에 길게 넣어줍니다.  

4. **말아주기**  
   - 김 끝쪽을 잡아 천천히 말아줍니다.  
   - 적당한 크기(약 2~3cm 두께)로 자릅니다.  

---

### 완성 및 서빙  
- 스시를 접시에 담아, 간장, 와사비, 생강절임과 함께 즐기세요!  

🇯🇵  
**맛있게 드세요!**
